In [2]:
import deeplabcut
import os
from deeplabcut.modelzoo import build_weight_init
import shutil
from modules.dlc_utils import set_transform_prob

superanimal_name = 'superanimal_quadruped'
project_path = 'projects/rat_pose'
config_path = os.path.join(project_path, "config.yaml")

Loading DLC 3.0.0rc13...


c:\Users\jiefei\anaconda3\envs\DEEPLABCUT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# initialize model weight

In [3]:
weight_init = build_weight_init(
            cfg = config_path,
            super_animal= superanimal_name,
            model_name='hrnet_w32',
            detector_name='fasterrcnn_resnet50_fpn_v2',
            with_decoder=False
)


# Create training data

In [4]:
## delete `training-datasets` folder
shuffle = 3
path1 = os.path.join(project_path, 'training-datasets/iteration-0/UnaugmentedDataSet_Sleap_Rat_testOct2')
if os.path.exists(path1):
    name_contain = f"shuffle{shuffle}"
    # delete everything that contains `shuffle{shuffle}` in the name
    for item in os.listdir(path1):
        if name_contain in item:
            os.remove(os.path.join(path1, item))

path2 = os.path.join(project_path, f'dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}')
if os.path.exists(path2):
    shutil.rmtree(path2)
    
dt = deeplabcut.create_training_dataset(config_path, Shuffles=[shuffle], weight_init=weight_init, net_type='hrnet_w32', userfeedback=False)

F:\code\pose_track\projects\rat_pose\labeled-data\RAT 11 FR1\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\RAT 11 FR1 10-02-25\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\RAT 2 FR1 10-02-25\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\RAT 4 FR1 10-02-25\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\RAT 6 FR1 10-03-25\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\RAT 8 FR1 10-03-25\CollectedData_rats.h5  not found (perhaps not annotated).
F:\code\pose_track\projects\rat_pose\labeled-data\Camera4_stitched - Trim\CollectedData_rats.h5  not found (perhaps not annotated).


# replace data augmentation parameters

In [5]:
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch
import yaml

loader = dlc_torch.DLCLoader(
    config=config_path,  
    trainset_index=0,
    shuffle=shuffle,
)

# Get the pytorch config
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"
model_cfg = read_config_as_dict(pytorch_config_path)
model_cfg

{'data': {'bbox_margin': 20,
  'colormode': 'RGB',
  'inference': {'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32}},
  'train': {'affine': {'p': 0.5,
    'rotation': 30,
    'scaling': [0.5, 1.25],
    'translation': 0},
   'crop_sampling': {'width': 448,
    'height': 448,
    'max_shift': 0.1,
    'method': 'hybrid'},
   'gaussian_noise': 12.75,
   'motion_blur': True,
   'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32}}},
 'device': 'auto',
 'inference': {'multithreading': {'enabled': True,
   'queue_length': 4,
   'timeout': 30.0},
  'compile': {'enabled': False, 'backend': 'inductor'},
  'autocast': {'enabled': False}},
 'metadata': {'project_path': 'F:\\code\\pose_track\\projects\\rat_pose',
  'pose_config_path': 'F:\\code\\pose_track\\projects\\rat_pose\\dlc-models-pytorch\\iteration-0\\Sleap_Rat_testOct2-trainset95shuffle3\\train\\pytorch_config.yaml',
  'bodyparts': ['head',
   'no

In [6]:
# repace train data params with dropin params
with open("projects/rat_pose/train_dropin.yaml", 'r') as f:
    dropin_param = yaml.safe_load(f)
dropin_param

{'crop_sampling': {'width': 448,
  'height': 448,
  'max_shift': 0.1,
  'method': 'hybrid'},
 'normalize_images': True,
 'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32},
 'transform': [{'augmentation': 'Affine',
   'p': 1,
   'rotate': [-30, 30],
   'scale': [0.5, 1.25],
   'translate_px': [0, 0],
   'keep_ratio': True},
  {'augmentation': 'GaussNoise',
   'var_limit': [0, 162.5625],
   'mean': 0.0,
   'per_channel': True,
   'p': 1},
  {'MotionBlur': None, 'p': 1},
  {'augmentation': 'ImageCompression',
   'quality_lower': 20,
   'quality_upper': 90,
   'p': 1},
  {'augmentation': 'RandomSunFlare',
   'p': 1,
   'flare_roi': [0.0, 0.0, 1.0, 1],
   'angle_lower': 0.0,
   'angle_upper': 1.0,
   'num_flare_circles_lower': 1,
   'num_flare_circles_upper': 4,
   'src_radius': 40,
   'src_color': [255, 245, 230]},
  {'augmentation': 'RandomRain',
   'slant_lower': -1,
   'slant_upper': 1,
   'drop_length': 15,
   'drop_width': 4,
   'drop_color': [200, 200, 200],
   'blu

In [7]:
model_cfg["data"]["train"] = dropin_param

model_cfg = set_transform_prob(model_cfg, prob=0.2)
n_trans = len(model_cfg["data"]["train"]['transform'])

# save
dlc_torch.config.write_config(pytorch_config_path,model_cfg)
model_cfg

{'data': {'bbox_margin': 20,
  'colormode': 'RGB',
  'inference': {'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32}},
  'train': {'crop_sampling': {'width': 448,
    'height': 448,
    'max_shift': 0.1,
    'method': 'hybrid'},
   'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32},
   'transform': [{'augmentation': 'Affine',
     'p': 0.2,
     'rotate': [-30, 30],
     'scale': [0.5, 1.25],
     'translate_px': [0, 0],
     'keep_ratio': True},
    {'augmentation': 'GaussNoise',
     'var_limit': [0, 162.5625],
     'mean': 0.0,
     'per_channel': True,
     'p': 0.2},
    {'MotionBlur': None, 'p': 0.2},
    {'augmentation': 'ImageCompression',
     'quality_lower': 20,
     'quality_upper': 90,
     'p': 0.2},
    {'augmentation': 'RandomSunFlare',
     'p': 0.2,
     'flare_roi': [0.0, 0.0, 1.0, 1],
     'angle_lower': 0.0,
     'angle_upper': 1.0,
     'num_flare_circles_lower': 1,
     '

# Train

In [8]:
# delete all pt files 
import glob
pt_files = glob.glob(f'projects/rat_pose/dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}/train/*.pt')
for f in pt_files:
    os.remove(f)

In [10]:
shuffle = 3
deeplabcut.train_network(
    config_path,
    shuffle=shuffle,
    epochs=200,
    save_epochs=10,
    superanimal_name=superanimal_name,
    batch_size= 16,
    keepdeconvweights=False,
    device="cuda:0",
    superanimal_transfer_learning=True
    )

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    auto_padding:
      pad_width_divisor: 32
      pad_height_divisor: 32
  train:
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    normalize_images: True
    auto_padding:
      pad_width_divisor: 32
      pad_height_divisor: 32
    transform: [{'augmentation': 'Affine', 'p': 0.2, 'rotate': [-30, 30], 'scale': [0.5, 1.25], 'translate_px': [0, 0], 'keep_ratio': True}, {'augmentation': 'GaussNoise', 'var_limit': [0, 162.5625], 'mean': 0.0, 'per_channel': True, 'p': 0.2}, {'MotionBlur': None, 'p': 0.2}, {'augmentation': 'ImageCompression', 'quality_lower': 20, 'quality_upper': 90, 'p': 0.2}, {'augmentation': 'RandomSunFlare', 'p': 0.2, 'flare_roi': [0.0, 0.0, 1.0, 1], 'angle_lower': 0.0, 'angle_upper': 1.0, 'num_flare_circles_lower': 1, 'num_flare_circles_upper': 4, 'src_radius': 40, 'src_color': [255, 245, 230]}, {'augmentation

KeyboardInterrupt: 